# CQR Score Smoke Test

This notebook checks the reviewer-requested CQR-score experiment path. It runs both nonnegative transformations described in the paper: capped scores and shifted scores.

In [9]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "utility").exists():
    raise RuntimeError(f"Run this notebook from the repository root, not {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

Repo root: e:\multi-target-scaling


In [40]:
from utility.exps import run_cqr_synthetic_experiment

methods = ["TSCP_R", "Unscaled", "TSCP_GWC"]

result = run_cqr_synthetic_experiment(
    dim_list=[3],
    sample_list=[30],
    alpha_list=[0.1],
    noise_type="Gaussian",
    noises_list=[[1, 5, 10]],
    trials=100,
    methods=methods,
    score_transform=["capped", "shifted"],
    shift_constant=20.0,
    base_interval_alpha=0.9,
    n_train_pool=2000,
    n_features=6,
    n_informative=6,
    oracle_n_samples=200,
    quantile_model_params={
        "n_estimators": 50,
        "max_depth": 2,
        "learning_rate": 0.08,
        "min_samples_leaf": 5,
    },
)

result.summary_results

,alpha,n_dim,n_cal,method,noise_type,score_type,score_transform,base_interval_alpha,shift_constant,n_trials,test_coverage_avg,test_coverage_1std,coverage_vol_avg,coverage_vol_1std,coverage_max_length_median,runtime_avg
0,0.1,3,30,TSCP_GWC,Gaussian,cqr,capped,0.9,0.0,100,0.907550,0.052605,52239.105537,25167.225107,52.792016,0.000140
1,0.1,3,30,TSCP_GWC,Gaussian,cqr,shifted,0.9,20.0,100,0.907975,0.052326,52225.956086,24929.988589,52.804904,0.000100
2,0.1,3,30,TSCP_R,Gaussian,cqr,capped,0.9,0.0,100,0.899100,0.051858,46839.141665,20109.837987,51.249127,0.000978
3,0.1,3,30,TSCP_R,Gaussian,cqr,shifted,0.9,20.0,100,0.907975,0.052326,52225.956086,24929.988589,52.804904,0.000808
4,0.1,3,30,Unscaled,Gaussian,cqr,capped,0.9,0.0,100,0.903325,0.048485,69944.106620,27218.814600,40.216181,0.000090
5,0.1,3,30,Unscaled,Gaussian,cqr,shifted,0.9,20.0,100,0.903325,0.048485,69944.106620,27218.814600,40.216181,0.000040


In [42]:
summary = result.summary_results
coord_summary = result.coordinate_summary_results

assert not summary.empty
assert not coord_summary.empty
assert set(summary["method"]) == set(methods)
assert set(summary["score_type"]) == {"cqr"}
assert set(summary["score_transform"]) == {"capped", "shifted"}
assert set(summary["base_interval_alpha"]) == {0.9}
assert set(coord_summary["coordinate"]) == {1, 2, 3}
assert (summary["test_coverage" if "test_coverage" in summary else "test_coverage_avg"] >= 0).all()
assert (coord_summary["coordinate_length_avg"] >= 0).all()

coord_summary.sort_values(["score_transform", "method", "coordinate"])

,alpha,n_dim,n_cal,method,noise_type,coordinate,score_type,score_transform,base_interval_alpha,shift_constant,n_trials,coordinate_length_avg,coordinate_length_1std,coordinate_length_median,coordinate_base_length_avg,coordinate_adjustment_avg
0,0.1,3,30,TSCP_GWC,Gaussian,1,cqr,capped,0.9,0.0,100,23.518655,6.502448,22.061974,1.554391,10.982132
2,0.1,3,30,TSCP_GWC,Gaussian,2,cqr,capped,0.9,0.0,100,39.520742,6.903773,39.575965,2.438471,18.541135
4,0.1,3,30,TSCP_GWC,Gaussian,3,cqr,capped,0.9,0.0,100,54.029359,9.271045,52.792016,2.982458,25.523450
6,0.1,3,30,TSCP_R,Gaussian,1,cqr,capped,0.9,0.0,100,22.730006,5.848968,21.107589,1.554391,10.587808
8,0.1,3,30,TSCP_R,Gaussian,2,cqr,capped,0.9,0.0,100,38.310515,6.303403,38.059611,2.438471,17.936022
10,0.1,3,30,TSCP_R,Gaussian,3,cqr,capped,0.9,0.0,100,52.365616,8.833350,51.249127,2.982458,24.691579
12,0.1,3,30,Unscaled,Gaussian,1,cqr,capped,0.9,0.0,100,39.767607,5.242434,38.853698,1.554391,19.106608
14,0.1,3,30,Unscaled,Gaussian,2,cqr,capped,0.9,0.0,100,40.651688,5.234210,39.874488,2.438471,19.106608
16,0.1,3,30,Unscaled,Gaussian,3,cqr,capped,0.9,0.0,100,41.195675,5.229512,40.216181,2.982458,19.106608
1,0.1,3,30,TSCP_GWC,Gaussian,1,cqr,shifted,0.9,20.0,100,23.523047,6.459296,22.198679,1.554391,10.984328


In [43]:
coord_summary.pivot_table(
    index=["score_transform", "coordinate"],
    columns="method",
    values="coordinate_length_avg",
)

method                       TSCP_GWC     TSCP_R   Unscaled
score_transform coordinate                                 
capped          1           23.518655  22.730006  39.767607
                2           39.520742  38.310515  40.651688
                3           54.029359  52.365616  41.195675
shifted         1           23.523047  23.523047  39.767607
                2           39.570974  39.570974  40.651688
                3           53.911474  53.911474  41.195675

For capped scores, `coordinate_adjustment_avg` should be nonnegative. For shifted scores, it may be negative because the final adjustment subtracts the shift constant, which corresponds to shrinking the original quantile interval on that coordinate.